# X-VC — преобразование голоса

Этот блокнот запускает официальный X-VC с простым Gradio-интерфейсом.

Запустите первую ячейку. Она проверит GPU, установит зависимости, заранее скачает checkpoint X-VC и вспомогательные модели и будет печатать понятные этапы с процентами. Проценты показывают завершённые этапы, а не выдуманную скорость загрузки. Если долгий этап продолжается, ячейка периодически сообщает, что именно она всё ещё делает.

После строки `100% — X-VC полностью подготовлена` запускайте вторую ячейку. Она только поднимает Gradio и сразу по ходу работы печатает состояние и публичную ссылку. В самом интерфейсе кнопка «Загрузить модель» уже не скачивает многогигабайтные файлы, а загружает подготовленные веса в GPU.


In [ ]:
import os, subprocess

print('0% — проверяю GPU', flush=True)
assert subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0, 'Включите GPU в настройках среды выполнения Colab.'
print('5% — GPU найден', flush=True)
print('7% — получаю свежий код X-VC Colab', flush=True)
subprocess.run(['rm', '-rf', '/content/audio-restoration-colab'], check=True)
clone = subprocess.run(
    ['git', 'clone', '--quiet', '--depth', '1', '--branch', 'agent/x-vc-colab', 'https://github.com/egor125552/audio-restoration-colab.git', '/content/audio-restoration-colab'],
    text=True,
    capture_output=True,
)
if clone.returncode != 0:
    print(clone.stderr, flush=True)
    raise RuntimeError('Не удалось скачать свежий код из GitHub.')
subprocess.run(
    ['python3', '-u', '/content/audio-restoration-colab/x_vc_colab/bootstrap_colab.py'],
    check=True,
)


In [ ]:
import re, subprocess
from IPython.display import Markdown, display

proc = subprocess.Popen(
    [
        'python3', '-u',
        '/content/audio-restoration-colab/x_vc_colab/launch_colab.py',
        '--python', '/content/x-vc/.venv/bin/python',
        '--port', '7860',
        '--timeout', '90',
        '--log', '/content/xvc-gradio.log',
        '--pid-file', '/content/xvc-gradio.pid',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
    match = re.search(r'XVC_PUBLIC_URL=(https://[^\s]+)', line)
    if match:
        public_url = match.group(1)

returncode = proc.wait()
if returncode != 0:
    raise RuntimeError('Не удалось запустить Gradio. Ошибка напечатана выше.')
if not public_url:
    raise RuntimeError('Launcher завершился без публичной ссылки.')
print('Публичная ссылка X-VC:', flush=True)
print(public_url, flush=True)
display(Markdown(f'[Открыть X-VC]({public_url})'))


In [ ]:
import os, signal
from pathlib import Path
pid_path = Path('/content/xvc-gradio.pid')
if pid_path.exists():
    try:
        os.kill(int(pid_path.read_text().strip()), signal.SIGTERM)
        pid_path.unlink(missing_ok=True)
        print('Интерфейс X-VC остановлен.')
    except ProcessLookupError:
        pid_path.unlink(missing_ok=True)
        print('Интерфейс уже остановлен.')
else:
    print('Запущенный интерфейс не найден.')
